# Active Learning Experiment

## Setup

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "agents").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from agents.al_agent import ActiveLearningAgent

annotated_path = PROJECT_ROOT / "data" / "labeled" / "annotated.parquet"
review_path = PROJECT_ROOT / "data" / "review_queue.csv"
config_path = PROJECT_ROOT / "config.yaml"

df_annotated = pd.read_parquet(annotated_path)
df_pool = pd.read_csv(review_path)
agent = ActiveLearningAgent(config_path=str(config_path))
review_label = str(agent._cfg.get("domain", {}).get("review_label", "other_or_offtopic"))
labeled_df = df_annotated.loc[
    df_annotated["label"].astype(str).ne("unlabeled")
    & df_annotated["label"].astype(str).ne(review_label)
].copy()

print({
    "annotated_rows": len(df_annotated),
    "labeled_thematic_rows": len(labeled_df),
    "pool_rows": len(df_pool),
    "classes": sorted(labeled_df["label"].astype(str).unique().tolist()),
})


## AL цикл

In [ ]:
entropy_history = agent.run_cycle(
    labeled_df=labeled_df,
    pool_df=df_pool,
    strategy="entropy",
)
entropy_df = pd.DataFrame(entropy_history)

fig_entropy = go.Figure()
fig_entropy.add_trace(
    go.Scatter(
        x=entropy_df["n_labeled"],
        y=entropy_df["accuracy"],
        mode="lines+markers",
        name="accuracy",
    )
)
fig_entropy.add_trace(
    go.Scatter(
        x=entropy_df["n_labeled"],
        y=entropy_df["f1_macro"],
        mode="lines+markers",
        name="f1_macro",
    )
)
fig_entropy.update_layout(
    title="AL цикл — entropy strategy",
    xaxis_title="Количество размеченных примеров",
    yaxis_title="Качество",
    template="plotly_white",
)
fig_entropy.show()
entropy_df


## Сравнение стратегий

In [ ]:
comparison = agent.compare_strategies(labeled_df=labeled_df, pool_df=df_pool)
comparison_rows = []
for strategy, history in comparison.items():
    for row in history:
        comparison_rows.append({**row, "strategy": strategy})
comparison_df = pd.DataFrame(comparison_rows)

fig_compare = px.line(
    comparison_df,
    x="n_labeled",
    y="f1_macro",
    color="strategy",
    markers=True,
    title="Entropy vs margin vs random",
    labels={"n_labeled": "Количество размеченных примеров", "f1_macro": "F1 macro"},
)
fig_compare.update_layout(template="plotly_white")
fig_compare.show()
comparison_df


## Вывод

In [ ]:
final_scores = {
    strategy: (history[-1]["f1_macro"] if history else 0.0)
    for strategy, history in comparison.items()
}
best_strategy = max(final_scores, key=final_scores.get)
max_quality = max(final_scores.values()) if final_scores else 0.0
target_quality = 0.9 * max_quality

n90 = {}
for strategy, history in comparison.items():
    history_df = pd.DataFrame(history)
    reached = history_df.loc[history_df["f1_macro"] >= target_quality, "n_labeled"]
    n90[strategy] = int(reached.iloc[0]) if not reached.empty else None

random_n = n90.get("random")
best_n = n90.get(best_strategy)
saved_examples = (
    int(random_n - best_n)
    if random_n is not None and best_n is not None
    else None
)

analysis = f"""
### Итоги эксперимента
- Лучшая стратегия по итоговому `f1_macro`: **{best_strategy}**.
- Относительно `random` лучшая стратегия экономит **{saved_examples if saved_examples is not None else 'н/д'}** примеров до достижения 90% от максимального качества.
- Порог **90% от max quality** достигается при **N={best_n if best_n is not None else 'не достигнуто'}** размеченных примерах.
- Финальные значения `f1_macro`: `entropy={final_scores.get('entropy', 0.0):.3f}`, `margin={final_scores.get('margin', 0.0):.3f}`, `random={final_scores.get('random', 0.0):.3f}`.
"""
display(Markdown(analysis))
